# Implementasi Bab 12 — Neural Network: MLP dengan Keras/TensorFlow

Notebook ini mengimplementasikan Praktikum 12.1–12.8 menggunakan dataset `load_digits`. Sumber utama: [ML-MODUL-01, hlm. 80–82].

> **Catatan adaptasi:** baris kode yang terpotong oleh tata letak PDF sudah disusun kembali menjadi Python valid. Model memakai layer `Input` eksplisit dan seed 42 agar lebih sesuai dengan API Keras modern dan lebih mudah direproduksi.

## Tujuan

1. Memuat dan membagi data angka tulisan tangan secara terstratifikasi.
2. Melakukan standardisasi tanpa data leakage.
3. Melatih MLP, mengevaluasi hasil, dan membaca kurva pelatihan.
4. Membandingkan arsitektur serta menerapkan early stopping.

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.datasets import load_digits
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)

## 1. Import dataset

Modul menggunakan `load_digits`, yaitu citra angka 0–9 berukuran 8×8 piksel [ML-MODUL-01, hlm. 81].

In [ ]:
digits = load_digits()
X, y = digits.data, digits.target

print("Ukuran X:", X.shape)
print("Ukuran y:", y.shape)
print("Kelas:", np.unique(y))

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for index, axis in enumerate(axes.ravel()):
    axis.imshow(digits.images[index], cmap="gray")
    axis.set_title(f"Label: {y[index]}")
    axis.axis("off")
plt.suptitle("Contoh Dataset Digits")
plt.tight_layout()
plt.show()

## 2. Split dan preprocessing

Data dibagi 80:20 dengan stratifikasi. `StandardScaler` hanya di-fit pada training set, lalu parameter yang sama dipakai untuk test set agar tidak terjadi data leakage [ML-MODUL-01, hlm. 81].

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.2,
    random_state=SEED,
    stratify=y_train_full,
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)
print("Test set:", X_test.shape)

## 3. Model MLP utama

Arsitektur mengikuti contoh modul: 64 neuron, dropout 0,3, 32 neuron, lalu output softmax 10 kelas [ML-MODUL-01, hlm. 81].

In [ ]:
model = keras.Sequential(
    [
        keras.layers.Input(shape=(X_train.shape[1],)),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dense(10, activation="softmax"),
    ],
    name="mlp_digits",
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    verbose=0,
)
print("Pelatihan model utama selesai.")

## 4. Evaluasi

Selain akurasi seperti pada modul, notebook menambahkan classification report dan confusion matrix agar kesalahan per kelas terlihat jelas. Tambahan ini merupakan pengetahuan umum/evaluasi pelengkap, bukan salinan materi Bab 12.

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
y_probability = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_probability, axis=1)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred, digits=4))

fig, axis = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    cmap="Blues",
    colorbar=False,
    ax=axis,
)
axis.set_title("Confusion Matrix — MLP Digits")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["loss"], label="Training Loss")
axes[0].plot(history.history["val_loss"], label="Validation Loss")
axes[0].set(title="Kurva Loss", xlabel="Epoch", ylabel="Loss")
axes[0].legend()

axes[1].plot(history.history["accuracy"], label="Training Accuracy")
axes[1].plot(history.history["val_accuracy"], label="Validation Accuracy")
axes[1].set(title="Kurva Accuracy", xlabel="Epoch", ylabel="Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()

### Interpretasi kurva

Jika training loss terus turun tetapi validation loss mulai naik, model mulai overfitting. Modul menyarankan early stopping atau regularisasi seperti dropout [ML-MODUL-01, hlm. 81].

## 5. Latihan: eksperimen arsitektur

Bandingkan hidden layer `[32]`, `[64, 32]`, dan `[128, 64, 32]` sebagaimana latihan modul [ML-MODUL-01, hlm. 82].

In [ ]:
def build_experiment_model(hidden_units):
    layers = [keras.layers.Input(shape=(X_train.shape[1],))]
    layers.extend(keras.layers.Dense(units, activation="relu") for units in hidden_units)
    layers.append(keras.layers.Dense(10, activation="softmax"))
    experiment_model = keras.Sequential(layers)
    experiment_model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return experiment_model


arsitektur_list = [[32], [64, 32], [128, 64, 32]]
architecture_results = []

for arsitektur in arsitektur_list:
    tf.keras.utils.set_random_seed(SEED)
    experiment_model = build_experiment_model(arsitektur)
    experiment_history = experiment_model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=30,
        batch_size=32,
        verbose=0,
    )
    final_validation_accuracy = experiment_history.history["val_accuracy"][-1]
    architecture_results.append((str(arsitektur), final_validation_accuracy))
    print(f"Arsitektur {arsitektur}: val_accuracy = {final_validation_accuracy:.4f}")

In [ ]:
architecture_names = [name for name, _ in architecture_results]
architecture_scores = [score for _, score in architecture_results]

plt.figure(figsize=(8, 4))
bars = plt.bar(architecture_names, architecture_scores, color="steelblue")
plt.ylim(0, 1)
plt.xlabel("Hidden layer")
plt.ylabel("Validation accuracy akhir")
plt.title("Perbandingan Arsitektur MLP")
plt.bar_label(bars, fmt="%.3f")
plt.tight_layout()
plt.show()

## 6. Latihan: early stopping

Callback menghentikan pelatihan apabila validation loss tidak membaik selama lima epoch dan mengembalikan bobot terbaik [ML-MODUL-01, hlm. 82]. Sebagai adaptasi, latihan ini membuat model `[64, 32]` baru agar eksperimen early stopping independen dan tidak melanjutkan bobot model dari eksperimen sebelumnya.

In [ ]:
tf.keras.utils.set_random_seed(SEED)
early_stopping_model = build_experiment_model([64, 32])
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
)

early_history = early_stopping_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0,
)

early_test_loss, early_test_accuracy = early_stopping_model.evaluate(
    X_test, y_test, verbose=0
)
print("Pelatihan berhenti pada epoch:", len(early_history.history["loss"]))
print(f"Test accuracy dengan early stopping: {early_test_accuracy:.4f}")

## Ringkasan

Notebook sudah mencakup Praktikum 12.1–12.8. Jalankan **Runtime → Run all** di Google Colab agar seluruh hasil dan grafik terbentuk berurutan. Nilai metrik dapat sedikit berbeda antarversi TensorFlow atau perangkat keras meskipun seed sudah ditetapkan.